# Comparação das planilhas dos layouts antigo e novo do CNJ

Análise dos dez arquivos `.xlsx` disponíveis em `output-antigo/` e `output-novo/` (TJAP, referência 05/2025). A comparação separa **valores**, **tipos das células**, **esquema tabular**, **formatação** e **estrutura interna do XLSX**.

## Conclusão executiva

As quatro planilhas de remuneração (`contracheque`, `direitos-eventuais`, `direitos-pessoais` e `indenizacoes`) preservam os mesmos valores, nas mesmas células e na mesma ordem, para os mesmos 82 magistrados. Portanto, não há indício de perda ou alteração dos valores de remuneração na migração do painel.

As exportações, porém, **não são integralmente iguais**. `controle-de-arquivos` perdeu duas linhas agregadas no layout novo, quatro cabeçalhos mudaram apenas a capitalização, a apresentação visual foi simplificada e a codificação interna do XLSX mudou. Em `direitos-eventuais`, 161 zeros que eram texto passaram a ser numéricos. Assim, a mudança é predominantemente estrutural/de exportação, mas não se limita à estrutura do site.

## Preparação do ambiente

Execute esta célula uma vez antes da análise. O uso de `%pip` garante que a dependência seja instalada no mesmo ambiente do kernel atual.

In [3]:
%pip install -q openpyxl


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
from collections import Counter, defaultdict
from hashlib import sha256
from pathlib import Path
from zipfile import ZipFile
import json

import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter

def localizar_raiz():
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / 'output-antigo').is_dir() and (base / 'output-novo').is_dir():
            return base.resolve()
    raise FileNotFoundError('Não foi possível localizar output-antigo/ e output-novo/.')

RAIZ = localizar_raiz()
DIR_ANTIGO = RAIZ / 'output-antigo'
DIR_NOVO = RAIZ / 'output-novo'
RAIZ

WindowsPath('C:/Users/gabri/OneDrive/Área de Trabalho/joao/TB/coletor-cnj')

## 1. Inventário e pareamento

Os nomes são usados como chave de pareamento. A célula falha explicitamente se existir arquivo sem correspondente no outro diretório.

In [5]:
antigos = {p.name: p for p in DIR_ANTIGO.glob('*.xlsx')}
novos = {p.name: p for p in DIR_NOVO.glob('*.xlsx')}
assert antigos.keys() == novos.keys(), {
    'somente_antigo': sorted(antigos.keys() - novos.keys()),
    'somente_novo': sorted(novos.keys() - antigos.keys()),
}
arquivos = sorted(antigos)
pd.DataFrame({'arquivo': arquivos, 'antigo': [antigos[n].stat().st_size for n in arquivos],
              'novo': [novos[n].stat().st_size for n in arquivos]})

,arquivo,antigo,novo
0,contracheque-tjap-2025-05.xlsx,16261,15562
1,controle-de-arquivos-tjap-2025-05.xlsx,5659,5578
2,direitos-eventuais-tjap-2025-05.xlsx,13191,12333
3,direitos-pessoais-tjap-2025-05.xlsx,10053,9637
4,indenizacoes-tjap-2025-05.xlsx,11780,11307


**Resultado apurado:** há cinco arquivos em cada diretório e os cinco nomes têm correspondente exato. Não há arquivo faltante ou adicional.

## 2. Funções de inspeção

A leitura usa `openpyxl` para não deixar o `pandas` renomear silenciosamente cabeçalhos duplicados. Valores são comparados de duas formas: (1) valor normalizado, para verificar o conteúdo lógico/visível; e (2) valor mais tipo, para detectar, por exemplo, texto `"0"` versus número `0`.

In [6]:
def tipo_logico(cell):
    if cell.value is None:
        return 'vazio'
    if cell.data_type == 'f':
        return 'formula'
    if cell.data_type == 'n':
        return 'numero'
    if cell.data_type == 'b':
        return 'booleano'
    if cell.data_type == 'e':
        return 'erro'
    return 'texto'

def normalizar_valor(v):
    if v is None:
        return ''
    if isinstance(v, float) and v.is_integer():
        return str(int(v))
    return str(v)

def inspecionar(path):
    wb = load_workbook(path, data_only=False, read_only=False)
    assert len(wb.sheetnames) == 1, f'{path.name}: esperado 1 sheet; obtido {wb.sheetnames}'
    ws = wb[wb.sheetnames[0]]
    celulas = [[ws.cell(r, c) for c in range(1, ws.max_column + 1)]
               for r in range(1, ws.max_row + 1)]
    valores = tuple(tuple(c.value for c in linha) for linha in celulas)
    normalizados = tuple(tuple(normalizar_valor(c.value) for c in linha) for linha in celulas)
    tipos = tuple(tuple(tipo_logico(c) for c in linha) for linha in celulas)
    cabecalhos = tuple(normalizados[0])
    repetidos = {k: v for k, v in Counter(cabecalhos).items() if v > 1}
    with ZipFile(path) as zf:
        partes = set(zf.namelist())
    bruto = path.read_bytes()
    return {
        'path': path, 'bytes': len(bruto), 'sha256': sha256(bruto).hexdigest(),
        'sheet': ws.title, 'dimensao': ws.calculate_dimension(),
        'linhas': ws.max_row, 'registros': max(ws.max_row - 1, 0), 'colunas': ws.max_column,
        'nao_vazias': sum(c.value is not None for linha in celulas for c in linha),
        'vazias_no_intervalo': sum(c.value is None for linha in celulas for c in linha),
        'cabecalhos': cabecalhos, 'cabecalhos_repetidos': repetidos,
        'valores': valores, 'normalizados': normalizados, 'tipos': tipos,
        'formulas': sum(c.data_type == 'f' for linha in celulas for c in linha),
        'mesclas': len(ws.merged_cells.ranges), 'filtro': ws.auto_filter.ref,
        'tabelas_excel': len(ws.tables),
        'estilos': dict(Counter(c.style_id for linha in celulas for c in linha)),
        'larguras': [ws.column_dimensions[get_column_letter(c)].width for c in range(1, ws.max_column + 1)],
        'alturas': dict(Counter(ws.row_dimensions[r].height or ws.sheet_format.defaultRowHeight
                                 for r in range(1, ws.max_row + 1))),
        'shared_strings': 'xl/sharedStrings.xml' in partes,
        'tipos_contagem': dict(Counter(t for linha in tipos for t in linha)),
    }

def diferenca_multiconjunto(linhas_a, linhas_b):
    a, b = Counter(linhas_a), Counter(linhas_b)
    return list((a - b).elements()), list((b - a).elements())

In [7]:
analises = {}
for nome in arquivos:
    antigo, novo = inspecionar(antigos[nome]), inspecionar(novos[nome])
    somente_antigo, somente_novo = diferenca_multiconjunto(antigo['normalizados'][1:], novo['normalizados'][1:])
    tipos_novos_por_linha = defaultdict(list)
    for valores, tipos in zip(novo['normalizados'][1:], novo['tipos'][1:]):
        tipos_novos_por_linha[valores].append(tipos)
    tipos_por_coluna = Counter()
    for valores, tipos_antigos in zip(antigo['normalizados'][1:], antigo['tipos'][1:]):
        if not tipos_novos_por_linha[valores]:
            continue
        tipos_novos = tipos_novos_por_linha[valores].pop()
        for i, (ta, tn) in enumerate(zip(tipos_antigos, tipos_novos), 1):
            if ta != tn:
                tipos_por_coluna[get_column_letter(i)] += 1
    analises[nome] = {
        'antigo': antigo, 'novo': novo,
        'valores_normalizados_iguais': antigo['normalizados'] == novo['normalizados'],
        'valores_e_tipos_iguais': (antigo['valores'], antigo['tipos']) == (novo['valores'], novo['tipos']),
        'cabecalhos_iguais': antigo['cabecalhos'] == novo['cabecalhos'],
        'somente_antigo': somente_antigo, 'somente_novo': somente_novo,
        'tipos_por_coluna': dict(tipos_por_coluna),
    }

## 3. Dimensões, conteúdo e tamanho dos arquivos

| Arquivo | Dimensão antiga | Dimensão nova | Registros antigo/novo | Colunas | Valores normalizados | Diferença de tamanho |
| --- | ---: | ---: | ---: | ---: | --- | ---: |
| contracheque | A1:R83 | A1:R83 | 82 / 82 | 18 | idênticos | -699 B (-4,30%) |
| controle-de-arquivos | A1:G4 | A1:G2 | 3 / 1 | 7 | diferentes | -81 B (-1,43%) |
| direitos-eventuais | A1:R83 | A1:R83 | 82 / 82 | 18 | idênticos | -858 B (-6,50%) |
| direitos-pessoais | A1:I83 | A1:I83 | 82 / 82 | 9 | idênticos | -416 B (-4,14%) |
| indenizacoes | A1:P83 | A1:P83 | 82 / 82 | 16 | idênticos | -473 B (-4,02%) |

Total: **56.944 bytes** no conjunto antigo e **54.417 bytes** no novo; redução de **2.527 bytes (4,44%)**. Os hashes SHA-256 físicos diferem em todos os cinco pares, como esperado diante das diferenças internas de estilo e serialização.

In [8]:
resumo = []
for nome, a in analises.items():
    velho, novo = a['antigo'], a['novo']
    resumo.append({
        'arquivo': nome, 'dimensão antiga': velho['dimensao'], 'dimensão nova': novo['dimensao'],
        'registros antigo': velho['registros'], 'registros novo': novo['registros'],
        'colunas antigo': velho['colunas'], 'colunas novo': novo['colunas'],
        'conteúdo normalizado igual': a['valores_normalizados_iguais'],
        'bytes antigo': velho['bytes'], 'bytes novo': novo['bytes'],
        'diferença bytes': novo['bytes'] - velho['bytes'],
    })
pd.DataFrame(resumo)

,arquivo,dimensão antiga,dimensão nova,registros antigo,registros novo,colunas antigo,colunas novo,conteúdo normalizado igual,bytes antigo,bytes novo,diferença bytes
0,contracheque-tjap-2025-05.xlsx,A1:R83,A1:R83,82,82,18,18,True,16261,15562,-699
1,controle-de-arquivos-tjap-2025-05.xlsx,A1:G4,A1:G2,3,1,7,7,False,5659,5578,-81
2,direitos-eventuais-tjap-2025-05.xlsx,A1:R83,A1:R83,82,82,18,18,True,13191,12333,-858
3,direitos-pessoais-tjap-2025-05.xlsx,A1:I83,A1:I83,82,82,9,9,True,10053,9637,-416
4,indenizacoes-tjap-2025-05.xlsx,A1:P83,A1:P83,82,82,16,16,True,11780,11307,-473


## 4. Diferença real em `controle-de-arquivos`

O layout antigo contém três linhas de dados; o novo contém apenas a linha detalhada. A linha preservada é exatamente:

`TJAP | 05/2025 | TJAP_05_25.xls 02/06/2025 09:14:30 | 82 | 1 | 1 | 1`

As duas linhas existentes somente no arquivo antigo são:

1. `Total | (vazio) | (vazio) | 82 | 1 | 1 | 1`
2. `TJAP | Total | (vazio) | 82 | 1 | 1 | 1`

Além disso, D:G mudaram somente a capitalização: `registros/arquivos/tribunais enviados/tribunais` passaram a `Registros/Arquivos/Tribunais Enviados/Tribunais`. Isso é uma alteração de esquema para consumidores que tratam nomes de colunas como *case-sensitive*.

In [9]:
controle = analises['controle-de-arquivos-tjap-2025-05.xlsx']
display(pd.DataFrame({
    'posição': [get_column_letter(i) for i in range(1, 8)],
    'antigo': controle['antigo']['cabecalhos'],
    'novo': controle['novo']['cabecalhos'],
}))
display(pd.DataFrame(controle['somente_antigo'], columns=controle['antigo']['cabecalhos']))

,posição,antigo,novo
0,A,Tribunal,Tribunal
1,B,Mês/Ano,Mês/Ano
2,C,Arquivo,Arquivo
3,D,Qtd. de registros,Qtd. de Registros
4,E,Qtd. de arquivos,Qtd. de Arquivos
5,F,Qtd. de tribunais enviados,Qtd. de Tribunais Enviados
6,G,Qtd. de tribunais,Qtd. de Tribunais


,Tribunal,Mês/Ano,Arquivo,Qtd. de registros,Qtd. de arquivos,Qtd. de tribunais enviados,Qtd. de tribunais
0,Total,,,82,1,1,1
1,TJAP,Total,,82,1,1,1


## 5. Colunas e ambiguidade de cabeçalhos

Nos quatro relatórios de remuneração, quantidade, nomes e ordem das colunas são iguais entre os layouts. Há, contudo, cabeçalhos duplicados já existentes nos dois layouts:

- `direitos-eventuais`: `Outra (R$)` em N/P e `Detalhe` em O/Q (duas ocorrências de cada).
- `direitos-pessoais`: `Outra (R$)` em E/G e `Detalhe` em F/H (duas ocorrências de cada).
- `indenizacoes`: `Outra (R$)` em J/L/N e `Detalhe` em K/M/O (três ocorrências de cada).

Isso não foi introduzido pelo painel novo, mas é uma limitação estrutural relevante: ao usar `pandas.read_excel`, nomes repetidos podem receber sufixos como `.1` e `.2`; selecionar apenas pelo texto original pode ser ambíguo. `contracheque` e `controle-de-arquivos` não têm cabeçalhos repetidos.

In [10]:
pd.DataFrame([
    {'arquivo': nome, 'cabeçalhos duplicados': a['antigo']['cabecalhos_repetidos'],
     'mesmos cabeçalhos': a['cabecalhos_iguais']}
    for nome, a in analises.items()
] )

,arquivo,cabeçalhos duplicados,mesmos cabeçalhos
0,contracheque-tjap-2025-05.xlsx,{},True
1,controle-de-arquivos-tjap-2025-05.xlsx,{},False
2,direitos-eventuais-tjap-2025-05.xlsx,"{'Outra (R$)': 2, 'Detalhe': 2}",True
3,direitos-pessoais-tjap-2025-05.xlsx,"{'Outra (R$)': 2, 'Detalhe': 2}",True
4,indenizacoes-tjap-2025-05.xlsx,"{'Outra (R$)': 3, 'Detalhe': 3}",True


## 6. Tipos das células

Em `contracheque`, `direitos-pessoais` e `indenizacoes`, as classes lógicas texto/número permanecem iguais célula a célula. Na linha detalhada preservada de `controle-de-arquivos`, também permanecem iguais.

A exceção é `direitos-eventuais`: **161 células com valor zero mudaram de texto para número** no arquivo novo:

- coluna F, `Antecipação de férias (R$)`: 79 células;
- coluna H, `Antecipação de gratificação natalina (R$)`: 82 células.

Os valores normalizados continuam sendo `0`, então somas e comparação numérica tendem a ficar mais consistentes no layout novo. Contudo, pipelines com validação rígida de tipo observarão a mudança.

In [11]:
pd.DataFrame([
    {'arquivo': nome, 'diferenças de tipo por coluna': a['tipos_por_coluna']}
    for nome, a in analises.items()
] )

,arquivo,diferenças de tipo por coluna
0,contracheque-tjap-2025-05.xlsx,{}
1,controle-de-arquivos-tjap-2025-05.xlsx,{}
2,direitos-eventuais-tjap-2025-05.xlsx,"{'F': 79, 'H': 82}"
3,direitos-pessoais-tjap-2025-05.xlsx,{}
4,indenizacoes-tjap-2025-05.xlsx,{}


## 7. Formatação e estrutura interna do XLSX

Diferenças presentes em todos os pares:

- antigo: textos armazenados como `inlineStr`; novo: tabela compartilhada `xl/sharedStrings.xml`;
- antigo: cabeçalho em Arial 8, negrito, preenchimento cinza, bordas, quebra de linha, larguras ajustadas por coluna e linhas mais altas;
- novo: aparência próxima do padrão Calibri 11, sem preenchimentos/bordas do exportador antigo, largura uniforme de aproximadamente 15,71 e altura padrão 15;
- antigo: nos quatro relatórios, cabeçalho com altura aproximada 39,22 e 82 linhas com 26,15; novo: todas as linhas com altura 15;
- nenhum arquivo contém fórmulas, células mescladas, autofiltro ou tabela estruturada do Excel; todos têm somente a aba `Sheet1`;
- os formatos numéricos e IDs de estilo foram reserializados. Isso e o uso de `sharedStrings` explicam os hashes e tamanhos físicos diferentes sem implicar alteração dos valores exibidos.

Portanto, igualdade binária (`SHA-256`) não é um critério adequado para validar a migração; a validação deve comparar esquema, valores normalizados e tipos lógicos.

In [12]:
estrutura = []
for nome, a in analises.items():
    for layout in ('antigo', 'novo'):
        x = a[layout]
        estrutura.append({
            'arquivo': nome, 'layout': layout, 'sheet': x['sheet'],
            'sharedStrings': x['shared_strings'], 'fórmulas': x['formulas'],
            'mesclas': x['mesclas'], 'autofiltro': bool(x['filtro']),
            'tabelas Excel': x['tabelas_excel'], 'estilos (id: quantidade)': x['estilos'],
            'alturas (altura: quantidade)': x['alturas'],
        })
pd.DataFrame(estrutura)

,arquivo,layout,sheet,sharedStrings,fórmulas,mesclas,autofiltro,tabelas Excel,estilos (id: quantidade),alturas (altura: quantidade)
0,contracheque-tjap-2025-05.xlsx,antigo,Sheet1,False,0,0,False,0,"{1: 4, 2: 1, 3: 13, 4: 410, 7: 820, 8: 246}","{39.221923828125: 1, 26.14794921875: 82}"
1,contracheque-tjap-2025-05.xlsx,novo,Sheet1,True,0,0,False,0,"{0: 415, 1: 1079}",{15.0: 83}
2,controle-de-arquivos-tjap-2025-05.xlsx,antigo,Sheet1,False,0,0,False,0,"{1: 2, 2: 1, 3: 4, 4: 1, 5: 2, 6: 2, 10: 8, 8:...",{26.14794921875: 4}
3,controle-de-arquivos-tjap-2025-05.xlsx,novo,Sheet1,True,0,0,False,0,"{0: 6, 2: 8}",{15.0: 2}
4,direitos-eventuais-tjap-2025-05.xlsx,antigo,Sheet1,False,0,0,False,0,"{1: 2, 2: 1, 3: 15, 4: 574, 6: 902}","{39.221923828125: 1, 26.14794921875: 82}"
5,direitos-eventuais-tjap-2025-05.xlsx,novo,Sheet1,True,0,0,False,0,"{0: 415, 1: 1079}",{15.0: 83}
6,direitos-pessoais-tjap-2025-05.xlsx,antigo,Sheet1,False,0,0,False,0,"{1: 2, 2: 1, 3: 6, 4: 410, 6: 328}","{39.221923828125: 1, 26.14794921875: 82}"
7,direitos-pessoais-tjap-2025-05.xlsx,novo,Sheet1,True,0,0,False,0,"{0: 415, 1: 332}",{15.0: 83}
8,indenizacoes-tjap-2025-05.xlsx,antigo,Sheet1,False,0,0,False,0,"{1: 2, 2: 1, 3: 13, 4: 492, 6: 820}","{39.221923828125: 1, 26.14794921875: 82}"
9,indenizacoes-tjap-2025-05.xlsx,novo,Sheet1,True,0,0,False,0,"{0: 498, 1: 830}",{15.0: 83}


## 8. Veredito

### Conclusão geral

Os layouts antigo e novo são **equivalentes quanto aos dados dos quatro relatórios de remuneração**, mas os dez arquivos XLSX não são integralmente iguais. A migração preservou os 82 magistrados, a ordem dos registros, os dados cadastrais e todos os valores monetários. As diferenças encontradas concentram-se no controle de arquivos, nos tipos de algumas células e na apresentação/serialização do Excel.

### Igualdade das colunas e do conteúdo

| Relatório | Registros antigo/novo | Colunas antigo/novo | Nomes e ordem das colunas | Valores | Observação |
| --- | ---: | ---: | --- | --- | --- |
| `contracheque` | 82 / 82 | 18 / 18 | Iguais | Idênticos célula a célula | Tipos lógicos também iguais |
| `direitos-eventuais` | 82 / 82 | 18 / 18 | Iguais | Idênticos após normalização | 161 zeros mudaram de texto para número |
| `direitos-pessoais` | 82 / 82 | 9 / 9 | Iguais | Idênticos célula a célula | Tipos lógicos também iguais |
| `indenizacoes` | 82 / 82 | 16 / 16 | Iguais | Idênticos célula a célula | Tipos lógicos também iguais |

Não houve inclusão, exclusão, renomeação ou deslocamento de colunas nesses quatro relatórios. Os cabeçalhos repetidos `Outra (R$)` e `Detalhe` continuam nas mesmas posições nos dois layouts; portanto, não são uma regressão do painel novo, embora permaneçam ambíguos para leitura automatizada.

Em `direitos-eventuais`, os 161 casos de mudança de tipo estão restritos a valores zero: 79 na coluna F (`Antecipação de férias (R$)`) e 82 na coluna H (`Antecipação de gratificação natalina (R$)`). O conteúdo visível não mudou, mas validadores rígidos podem distinguir texto `"0"` de número `0`.

### Observações sobre `controle-de-arquivos`

O controle de arquivos merece tratamento separado porque **não possui a mesma quantidade de registros**:

- ambos os layouts possuem sete colunas e mantêm a mesma ordem funcional;
- o intervalo mudou de `A1:G4` para `A1:G2`;
- o arquivo antigo possui três registros e o novo possui apenas um;
- a linha detalhada `TJAP | 05/2025 | TJAP_05_25.xls 02/06/2025 09:14:30 | 82 | 1 | 1 | 1` foi preservada integralmente;
- foram removidas somente as linhas agregadas `Total` geral e `TJAP | Total`;
- os cabeçalhos D:G mantêm o mesmo significado, mas passaram de `Qtd. de registros`, `Qtd. de arquivos`, `Qtd. de tribunais enviados` e `Qtd. de tribunais` para as versões com iniciais maiúsculas `Registros`, `Arquivos`, `Tribunais Enviados` e `Tribunais`.

Logo, não houve perda do registro detalhado que comprova o envio do TJAP, mas houve perda das linhas de consolidação exibidas pelo painel antigo. Consumidores que dependem dessas totalizações deverão recalculá-las ou adaptar a leitura. Código que acessa as colunas pelo nome exato também deverá considerar a mudança de capitalização.

### Diferenças que não representam mudança de dados

Os arquivos novos são menores porque utilizam strings compartilhadas, menos estilos e formatação visual simplificada. Todos continuam com uma única aba `Sheet1` e sem fórmulas, mesclas, autofiltros ou tabelas estruturadas do Excel. Diferenças de tamanho, hash, fonte, bordas, largura ou altura não indicam perda de conteúdo.

### Decisão final

Para coleta e análise dos dados remuneratórios, o resultado do painel novo pode ser considerado equivalente ao antigo. Não é correto, contudo, afirmar que houve *somente* mudança na estrutura visual do site: a exportação também mudou a estrutura do `controle-de-arquivos`, a capitalização de quatro cabeçalhos, o tipo de 161 zeros e a codificação/formatação interna dos XLSX. Os testes automatizados devem validar valores normalizados, nomes e ordem das colunas, quantidade de registros e tipos lógicos relevantes, sem exigir igualdade binária dos arquivos.